# Deteksi Tuberkulosis dari Citra X-ray — Pipeline Lengkap
### Untuk paper J. ICT Res. Appl.

**Cara pakai:** jalankan sel dari atas ke bawah satu per satu (Shift+Enter), atau menu **Runtime > Run all**.

**WAJIB sebelum mulai:** aktifkan GPU lewat **Runtime > Change runtime type > T4 GPU**.

Notebook ini membandingkan 5 arsitektur transfer learning, menguji generalisasi lintas-dataset, menganalisis trade-off ukuran model, dan membuat visualisasi Grad-CAM. Semua hasil tersimpan ke folder `outputs/` dan disalin ke Google Drive Anda.


## 1. Cek GPU
Pastikan baris ini menampilkan Tesla T4 atau GPU lain. Jika kosong, aktifkan GPU dulu (Runtime > Change runtime type).

In [ ]:
!nvidia-smi -L

## 2. Unduh dataset dari Kaggle
Dataset ini publik, jadi **tidak perlu token**. Cukup jalankan.

In [ ]:
!pip install -q kaggle
!kaggle datasets download -d tawsifurrahman/tuberculosis-tb-chest-xray-dataset
!unzip -q -o tuberculosis-tb-chest-xray-dataset.zip -d data_raw
print("Selesai unduh & ekstrak.")

## 3. Tata folder
Memindahkan folder gambar ke struktur yang dibutuhkan pipeline: `data/main/Normal` dan `data/main/Tuberculosis`.

In [ ]:
import shutil, os
src = "data_raw/TB_Chest_Radiography_Database"
os.makedirs("data/main", exist_ok=True)
for cls in ["Normal", "Tuberculosis"]:
    dst = f"data/main/{cls}"
    if not os.path.isdir(dst):
        shutil.move(f"{src}/{cls}", dst)
print("Isi data/main:", os.listdir("data/main"))
print("Normal:", len(os.listdir("data/main/Normal")))
print("Tuberculosis:", len(os.listdir("data/main/Tuberculosis")))

## 4. Konfigurasi & import
Atur parameter di sini. Untuk **uji coba cepat**, biarkan `MODELS_TO_RUN` berisi satu model dulu. Setelah yakin lancar, tambahkan model lain.

In [ ]:
import os, time, numpy as np, pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import (
    ResNet50, EfficientNetB0, MobileNetV2, DenseNet121, VGG16,
    resnet50, efficientnet, mobilenet_v2, densenet, vgg16)
from sklearn.metrics import (confusion_matrix, roc_curve, auc,
    precision_recall_fscore_support, accuracy_score)

# ---- Konfigurasi ----
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
EPOCHS_HEAD, EPOCHS_FT = 8, 12
LR_HEAD, LR_FT = 1e-3, 1e-5
SEED = 42
VAL_SPLIT, TEST_SPLIT = 0.15, 0.15
OUT_DIR = "outputs"
DATA_MAIN = "data/main"
DATA_EXTERNAL = None   # set "data/external" bila punya Montgomery+Shenzhen

# Untuk uji coba: mulai dengan 1 model. Tambahkan nama lain bila sudah lancar.
MODELS_TO_RUN = ["MobileNetV2"]
# Pilihan lengkap: ["ResNet50","EfficientNetB0","MobileNetV2","DenseNet121","VGG16"]

ALL_MODELS = {
    "ResNet50":       (ResNet50,       resnet50.preprocess_input,     "conv5_block3_out"),
    "EfficientNetB0": (EfficientNetB0, efficientnet.preprocess_input, "top_conv"),
    "MobileNetV2":    (MobileNetV2,    mobilenet_v2.preprocess_input, "out_relu"),
    "DenseNet121":    (DenseNet121,    densenet.preprocess_input,     "relu"),
    "VGG16":          (VGG16,          vgg16.preprocess_input,        "block5_conv3"),
}
tf.random.set_seed(SEED); np.random.seed(SEED)
os.makedirs(OUT_DIR, exist_ok=True)
print("TF:", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

## 5. Fungsi data, model, evaluasi, plot, Grad-CAM
Jalankan sel ini sekali (mendefinisikan semua fungsi). Tidak ada output yang diharapkan.

In [ ]:
def build_datasets(preprocess):
    full = tf.keras.utils.image_dataset_from_directory(
        DATA_MAIN, labels="inferred", label_mode="binary",
        class_names=["Normal","Tuberculosis"], image_size=IMG_SIZE,
        batch_size=None, shuffle=True, seed=SEED)
    n = full.cardinality().numpy()
    n_test, n_val = int(n*TEST_SPLIT), int(n*VAL_SPLIT)
    test_ds = full.take(n_test); rest = full.skip(n_test)
    val_ds = rest.take(n_val);   train_ds = rest.skip(n_val)
    aug = tf.keras.Sequential([
        layers.RandomFlip("horizontal"), layers.RandomRotation(0.05),
        layers.RandomZoom(0.10), layers.RandomContrast(0.10)])
    AT = tf.data.AUTOTUNE
    def prep(img, lab, tr):
        img = preprocess(tf.cast(img, tf.float32))
        if tr: img = aug(img, training=True)
        return img, lab
    train_ds = train_ds.map(lambda x,y: prep(x,y,True), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    val_ds   = val_ds.map(lambda x,y: prep(x,y,False), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    test_ds  = test_ds.map(lambda x,y: prep(x,y,False), num_parallel_calls=AT).batch(BATCH_SIZE).prefetch(AT)
    return train_ds, val_ds, test_ds

def build_model(builder):
    base = builder(include_top=False, weights="imagenet",
                   input_shape=IMG_SIZE+(3,), pooling="avg")
    base.trainable = False
    inp = tf.keras.Input(shape=IMG_SIZE+(3,))
    x = base(inp, training=False)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(128, activation="relu")(x)
    x = layers.Dropout(0.2)(x)
    out = layers.Dense(1, activation="sigmoid")(x)
    return models.Model(inp, out), base

def evaluate(model, ds):
    yt, yp = [], []
    for xb, yb in ds:
        yp.extend(model.predict(xb, verbose=0).ravel().tolist())
        yt.extend(yb.numpy().ravel().tolist())
    yt, yp = np.array(yt), np.array(yp)
    pred = (yp>=0.5).astype(int)
    acc = accuracy_score(yt, pred)
    pr, rc, f1, _ = precision_recall_fscore_support(yt, pred, average="binary", zero_division=0)
    fpr, tpr, _ = roc_curve(yt, yp); a = auc(fpr, tpr)
    return {"accuracy":acc,"precision":pr,"recall":rc,"f1":f1,"auc":a}, yt, yp

def measure_inference(model, n=50):
    d = tf.random.normal((1,)+IMG_SIZE+(3,)); model.predict(d, verbose=0)
    t0=time.time()
    for _ in range(n): model.predict(d, verbose=0)
    return (time.time()-t0)/n*1000

def plot_history(name,h1,h2):
    acc=h1.history["accuracy"]+h2.history["accuracy"]
    val=h1.history["val_accuracy"]+h2.history["val_accuracy"]
    plt.figure(figsize=(6,4)); plt.plot(acc,label="train"); plt.plot(val,label="val")
    plt.axvline(len(h1.history["accuracy"])-0.5, ls="--", c="gray")
    plt.xlabel("Epoch"); plt.ylabel("Accuracy"); plt.title(f"History {name}"); plt.legend()
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/history_{name}.png", dpi=150); plt.close()

def plot_cm(name, yt, yp):
    cm = confusion_matrix(yt, (yp>=0.5).astype(int))
    plt.figure(figsize=(4.5,4)); plt.imshow(cm, cmap="Blues"); plt.colorbar()
    plt.title(f"Confusion Matrix {name}")
    plt.xticks([0,1],["Normal","TB"]); plt.yticks([0,1],["Normal","TB"])
    plt.xlabel("Predicted"); plt.ylabel("True")
    for i in range(2):
        for j in range(2):
            plt.text(j,i,str(cm[i,j]),ha="center",va="center",
                     color="white" if cm[i,j]>cm.max()/2 else "black")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/cm_{name}.png", dpi=150); plt.close()

def gradcam(model, preprocess, name, last_conv, sample):
    if not sample or not os.path.isfile(sample): return
    img = tf.keras.utils.load_img(sample, target_size=IMG_SIZE)
    arr = tf.keras.utils.img_to_array(img)
    x = preprocess(np.expand_dims(arr.copy(),0))
    base = model.layers[1]
    gm = models.Model(base.inputs, [base.get_layer(last_conv).output, base.output])
    with tf.GradientTape() as tape:
        conv, feat = gm(x); tape.watch(conv)
        # alirkan feat (sudah pooled) ke head
        h = feat
        for L in model.layers[2:]:
            h = L(h)
        pred = h
    grads = tape.gradient(pred, conv)
    pooled = tf.reduce_mean(grads, axis=(0,1,2))
    conv = conv[0]
    heat = tf.squeeze(conv @ pooled[...,None])
    heat = tf.maximum(heat,0)/(tf.reduce_max(heat)+1e-8)
    heat = heat.numpy()
    up = np.kron(heat, np.ones((IMG_SIZE[0]//heat.shape[0]+1, IMG_SIZE[1]//heat.shape[1]+1)))[:IMG_SIZE[0],:IMG_SIZE[1]]
    plt.figure(figsize=(8,4))
    plt.subplot(1,2,1); plt.imshow(arr.astype("uint8")); plt.axis("off"); plt.title("Input")
    plt.subplot(1,2,2); plt.imshow(arr.astype("uint8")); plt.imshow(up, cmap="jet", alpha=0.45)
    plt.axis("off"); plt.title(f"Grad-CAM {name}")
    plt.tight_layout(); plt.savefig(f"{OUT_DIR}/gradcam_{name}.png", dpi=150); plt.close()

def find_tb_sample():
    d = os.path.join(DATA_MAIN,"Tuberculosis")
    for f in os.listdir(d):
        if f.lower().endswith((".png",".jpg",".jpeg")): return os.path.join(d,f)
    return ""
print("Semua fungsi siap.")

## 6. Training & evaluasi
Inilah sel utama. Untuk tiap model: latih (2 tahap), evaluasi, simpan confusion matrix + Grad-CAM + history.

**Class weight** otomatis dihitung untuk mengatasi ketidakseimbangan kelas (3501 Normal vs 701 TB).

In [ ]:
rows, roc_data, tradeoff, cross = [], {}, [], []
sample = find_tb_sample()
class_weight = {0: 1.0, 1: 3501/701}  # bobot lebih besar untuk TB (minoritas)

for name in MODELS_TO_RUN:
    builder, preprocess, last_conv = ALL_MODELS[name]
    print(f"\n{'='*55}\n  {name}\n{'='*55}")
    train_ds, val_ds, test_ds = build_datasets(preprocess)
    model, base = build_model(builder)
    es = callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True)

    model.compile(optimizers.Adam(LR_HEAD), "binary_crossentropy", metrics=["accuracy"])
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD,
                   callbacks=[es], class_weight=class_weight, verbose=2)

    base.trainable = True
    for L in base.layers[:-30]: L.trainable = False
    model.compile(optimizers.Adam(LR_FT), "binary_crossentropy", metrics=["accuracy"])
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT,
                   callbacks=[es], class_weight=class_weight, verbose=2)

    plot_history(name, h1, h2)
    m, yt, yp = evaluate(model, test_ds)
    rows.append({"model":name, **{k:round(v,4) for k,v in m.items()}})
    roc_data[name] = (yt, yp)
    plot_cm(name, yt, yp)
    tradeoff.append({"model":name, "accuracy":round(m["accuracy"],4),
                     "params_millions":round(model.count_params()/1e6,2),
                     "inference_ms":round(measure_inference(model),2)})
    try: gradcam(model, preprocess, name, last_conv, sample)
    except Exception as e: print("Grad-CAM gagal:", e)
    print(f"  -> acc={m['accuracy']:.4f} recall(TB)={m['recall']:.4f} f1={m['f1']:.4f} auc={m['auc']:.4f}")
    tf.keras.backend.clear_session()

print("\nTraining selesai.")
pd.DataFrame(rows)

## 7. Simpan tabel & kurva ROC

In [ ]:
pd.DataFrame(rows).to_csv(f"{OUT_DIR}/results_table.csv", index=False)
pd.DataFrame(tradeoff).to_csv(f"{OUT_DIR}/tradeoff.csv", index=False)

plt.figure(figsize=(6,5))
for name,(yt,yp) in roc_data.items():
    fpr,tpr,_ = roc_curve(yt,yp); plt.plot(fpr,tpr,label=f"{name} (AUC={auc(fpr,tpr):.3f})")
plt.plot([0,1],[0,1],"k--",alpha=0.4)
plt.xlabel("FPR"); plt.ylabel("TPR"); plt.title("ROC Comparison"); plt.legend(loc="lower right")
plt.tight_layout(); plt.savefig(f"{OUT_DIR}/roc_comparison.png", dpi=150); plt.show()

print("Tabel hasil:"); display(pd.DataFrame(rows))
print("Trade-off:");  display(pd.DataFrame(tradeoff))

## 8. Lihat hasil visual (confusion matrix, Grad-CAM)

In [ ]:
from IPython.display import Image, display
for name in MODELS_TO_RUN:
    for kind in ["cm","gradcam","history"]:
        p = f"{OUT_DIR}/{kind}_{name}.png"
        if os.path.isfile(p):
            print(f"--- {kind} {name} ---"); display(Image(p))

## 9. Simpan semua hasil ke Google Drive
Agar hasil tidak hilang saat sesi Colab berakhir.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import shutil
dest = "/content/drive/MyDrive/tb_outputs"
shutil.rmtree(dest, ignore_errors=True)
shutil.copytree("outputs", dest)
print("Tersimpan ke:", dest)

---
### Setelah uji coba berhasil
Kalau sel 6 selesai tanpa error dengan satu model, kembali ke **sel 4** dan ganti:
```python
MODELS_TO_RUN = ["ResNet50","EfficientNetB0","MobileNetV2","DenseNet121","VGG16"]
```
lalu jalankan ulang sel 6-9 untuk membandingkan semua arsitektur.

**Catatan interpretasi untuk paper:** karena data tidak seimbang, perhatikan **recall pada kelas TB** (jangan sampai pasien TB lolos terdeteksi), bukan hanya accuracy.